# OpenLearn-AI OCR Benchmark — External GPU Smoke Validation

Validation harness only. The benchmark runs **unchanged**; Colab is an execution
environment, not a benchmark layer.

Flow: pinned GitHub checkout → uv-managed Python 3.12 `.venv` → canonical Misraj
archive from Drive (SHA-256 gated) → existing CLI (`ocrbench.run.run_text`) →
validated timestamped result → archived back to Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Configuration

In [3]:
import subprocess
from pathlib import Path

# --- pinned inputs (change here only) -------------------------------------
REPO_URL       = "https://github.com/MuhammadSeyam/OpenLearn-AI.git"
PINNED_COMMIT  = "2c6975c3d5b212d7c958cb77a2bc1065726dd363"
ARCHIVE_NAME   = "ocrbench-misraj-data-v1.tar.gz"
ARCHIVE_SHA256 = "b66f8e9af44197bf65c2ee0f1c684744e16495390cf87c9fada7fc76af03f7b0"
LIMIT          = 20
# --------------------------------------------------------------------------

REPO    = Path("/content/OpenLearn-AI")
BENCH   = REPO / "experiments/OCR/ocr-benchmark"
VENV_PY = BENCH / ".venv/bin/python"
DRIVE   = Path("/content/drive/MyDrive/ocrbench")
DIST    = DRIVE / "dist"
ARCHIVE = DIST / ARCHIVE_NAME
RESULTS = BENCH / "results/formal/misraj/docling"
MANIFEST = BENCH / "configs/datasets/misraj_DATA_MANIFEST.sha256"

def sh(cmd: str, cwd: Path | None = None) -> str:
    """Run a shell command deterministically; raise on non-zero exit."""
    r = subprocess.run(cmd, shell=True, cwd=cwd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{r.stderr[-2000:]}")
    return r.stdout.strip()

report: dict[str, object] = {}
print("configuration ready")

configuration ready


## 1. Mount Drive

In [4]:
from google.colab import drive
drive.mount("/content/drive")
assert ARCHIVE.is_file(), f"archive missing in Drive: {ARCHIVE}"
sidecar = DIST / f"{ARCHIVE_NAME}.sha256"
if sidecar.is_file():
    assert sidecar.read_text().split()[0] == ARCHIVE_SHA256, "Drive sidecar hash mismatch"
print("Drive archive present:", ARCHIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive archive present: /content/drive/MyDrive/ocrbench/dist/ocrbench-misraj-data-v1.tar.gz


## 2. GPU / CUDA Gate

Environment-level gate on the host runtime. CPU fallback is forbidden.

In [5]:
import shutil, torch
assert shutil.which("nvidia-smi"), "no NVIDIA runtime attached — select a GPU runtime"
print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"))
assert torch.cuda.is_available(), "CUDA unavailable — GPU-first violation"
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
x = torch.randn(512, 512, device="cuda")
assert float((x @ x).sum()) != 0.0, "CUDA tensor op failed"
print(f"GPU {GPU_NAME} | VRAM {props.total_memory/1e9:.1f} GB | "
      f"cc {props.major}.{props.minor} | host torch {torch.__version__} "
      f"(cuda {torch.version.cuda})")

Tesla T4, 15360 MiB, 580.82.07
GPU Tesla T4 | VRAM 15.6 GB | cc 7.5 | host torch 2.11.0+cu128 (cuda 12.8)


## 3. Repository Checkout (pinned commit, rerun-safe)

In [6]:
if REPO.is_dir():
    sh("git fetch origin", cwd=REPO)
else:
    sh(f"git clone {REPO_URL} {REPO}", cwd=Path("/content"))
sh(f"git checkout {PINNED_COMMIT}", cwd=REPO)
head = sh("git rev-parse HEAD", cwd=REPO)
if head != PINNED_COMMIT:
    raise SystemExit(f"FAIL: HEAD {head} != pinned {PINNED_COMMIT}")
report["Repository commit"] = "PASS"
print("checked out:", head)

checked out: 2c6975c3d5b212d7c958cb77a2bc1065726dd363


## 4. Repository Verification

In [7]:
required = [
    "src/ocrbench", "src/ocrbench/datasets", "src/ocrbench/engines",
    "src/ocrbench/metrics", "src/ocrbench/run", "src/ocrbench/types.py",
    "pyproject.toml", "uv.lock",
    "configs/datasets/misraj_DATA_MANIFEST.sha256",
]
missing = [rel for rel in required if not (BENCH / rel).exists()]
assert not missing, f"missing in checkout: {missing}"
print("repository structure verified:", len(required), "paths present")

repository structure verified: 9 paths present


## 5. Python 3.12 Environment

`--clear` keeps reruns deterministic (no interactive prompts); the host
Python version is irrelevant.

In [8]:
sh("pip install -q uv")
print("uv:", sh("uv --version"))
sh("uv python install 3.12", cwd=BENCH)
sh("uv venv --clear --python 3.12 .venv", cwd=BENCH)
pyver = sh(f"{VENV_PY} --version")
assert pyver.split()[1].startswith("3.12."), f"expected 3.12.x, got {pyver}"
report["Python 3.12 environment"] = pyver
print(".venv:", pyver)

uv: uv 0.12.6 (x86_64-unknown-linux-gnu)
.venv: Python 3.12.14


## 6. Dependency Installation

`uv sync` operates strictly on the project `.venv`. The ad-hoc Docling
install passes an explicit interpreter (`--python .venv/bin/python`) so it
can never leak into the host Python (lesson from the previous notebook).

In [9]:
sh("uv sync --frozen", cwd=BENCH)
out = sh(f'uv pip install --python {VENV_PY} "docling>=2.0"', cwd=BENCH)
print(out.splitlines()[-1] if out else "docling already satisfied")
installed = sh(f"uv pip list --python {VENV_PY}")
for pkg in ("ocrbench", "docling", "torch", "pyarrow"):
    line = [l for l in installed.splitlines() if l.lower().startswith(pkg + " ")]
    assert line, f"{pkg} missing from .venv"
    print("in .venv:", line[0])

docling already satisfied
in .venv: ocrbench                  0.1.0       /content/OpenLearn-AI/experiments/OCR/ocr-benchmark
in .venv: docling                   2.122.0
in .venv: torch                     2.13.0
in .venv: pyarrow                   25.0.1


## 7. Environment Verification (inside `.venv` only)

In [11]:
verify_script = """
import sys
import torch
import docling
import ocrbench
import ocrbench.run.run_text

ver = sys.version.split()[0]
assert ver.startswith("3.12."), f"wrong python in .venv: {ver}"
assert torch.cuda.is_available(), "CUDA unavailable inside .venv"

print("python:", ver)
print(
    "torch:", torch.__version__,
    "| cuda build:", torch.version.cuda,
    "| available:", torch.cuda.is_available()
)
print("docling:", docling.__version__)
print("ocrbench.run.run_text import OK")
"""

verify_file = Path("/tmp/ocrbench_verify.py")
verify_file.write_text(verify_script)

print(sh(f"{VENV_PY} {verify_file}"))
report["Torch CUDA"] = "PASS"
report["Docling"] = "PASS"

python: 3.12.14
torch: 2.13.0+cu130 | cuda build: 13.0 | available: True
docling: 2.122.0
ocrbench.run.run_text import OK


## 8. Dataset Archive Verification

In [12]:
import hashlib
digest = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
print("computed:", digest)
if digest != ARCHIVE_SHA256:
    raise SystemExit(f"FAIL: archive hash mismatch\n got {digest}\n want {ARCHIVE_SHA256}")
report["Dataset archive hash"] = "PASS"
print("archive hash verified")

computed: b66f8e9af44197bf65c2ee0f1c684744e16495390cf87c9fada7fc76af03f7b0
archive hash verified


## 9. Dataset Extraction + Integrity Verification

The tracked manifest is authoritative: every listed file is checked for
existence, byte size, and SHA-256. No filenames are hard-coded here.

In [13]:
sh(f"tar -xzf {ARCHIVE} -C {BENCH}")
lines = MANIFEST.read_text().splitlines()
assert lines, "empty manifest"
for line in lines:
    expected_hash, expected_size, rel = line.split()
    f = BENCH / rel
    assert f.is_file(), f"missing after extraction: {rel}"
    assert f.stat().st_size == int(expected_size), f"size mismatch: {rel}"
    h = hashlib.sha256(f.read_bytes()).hexdigest()
    if h != expected_hash:
        raise SystemExit(f"FAIL: hash mismatch for {rel}")
    print("verified:", rel, f"({expected_size} bytes)")
report["Dataset manifest integrity"] = f"PASS ({len(lines)} files)"
print("dataset integrity PASSED")

verified: data/processed/misraj/data/train-00000-of-00002.parquet (247992106 bytes)
verified: data/processed/misraj/data/train-00001-of-00002.parquet (289044035 bytes)
dataset integrity PASSED


## 10. Benchmark Preflight

In [16]:
# 1) .venv python is 3.12
assert sh(f"{VENV_PY} --version").split()[1].startswith("3.12.")

# 2-6) imports + CUDA from the SAME .venv, including the runner module
preflight_script = """
import sys
import torch
import docling
import ocrbench
import ocrbench.run.run_text

assert sys.version.split()[0].startswith("3.12.")
assert torch.cuda.is_available()

print(".venv preflight imports OK; cuda available")
"""

preflight_file = Path("/tmp/ocrbench_preflight.py")
preflight_file.write_text(preflight_script)

print(sh(f"{VENV_PY} {preflight_file}"))

# 7) dataset files listed by the manifest exist
for line in MANIFEST.read_text().splitlines():
    assert (BENCH / line.split()[2]).is_file()

# 8) result root can be created
RESULTS.mkdir(parents=True, exist_ok=True)
existing_runs = {p.name for p in RESULTS.iterdir() if p.is_dir()}

print("PREFLIGHT PASSED")

.venv preflight imports OK; cuda available
PREFLIGHT PASSED


## 11. Benchmark Execution

The existing CLI, unchanged. Result directories are snapshotted before the
run so the newly created one is identified unambiguously afterwards.

In [17]:
run_log = sh(
    f"{VENV_PY} -m ocrbench.run.run_text --limit {LIMIT} --engine docling",
    cwd=BENCH,
)
print(run_log)
new_runs = {p.name for p in RESULTS.iterdir() if p.is_dir()} - existing_runs
assert len(new_runs) == 1, f"expected exactly one new result directory, got {sorted(new_runs)}"
run_dir = RESULTS / new_runs.pop()
report["Benchmark execution"] = "PASS"
print("new result directory:", run_dir)

[run] loading misraj...
[run] loading docling (GPU)...
[run] warm-up done (38.4s)
[run 1/20] 00f75222-bc17-4ce8-b7ae-3bc9a1ce8e70: ok=True cer_norm=0.9961146187469645
[run 2/20] 015eec69-bc03-49cf-a021-13aa50b27343: ok=True cer_norm=0.9271255060728745
[run 3/20] 01e9cad5-acc1-4b9c-8054-1e76224bfbc6: ok=True cer_norm=0.9771380186282811
[run 4/20] 0235e7ae-9593-423f-8dcd-49788841d5b8: ok=True cer_norm=0.993006993006993
[run 5/20] 0245a517-cac2-43c3-9cda-d53e8c336f27: ok=True cer_norm=0.997384481255449
[run 6/20] 03d2cd12-6353-4527-a3a2-879b6c53e367: ok=True cer_norm=0.8746268656716418
[run 7/20] 040e0cfb-8182-49ac-8546-01c051948068: ok=True cer_norm=0.9775828460038987
[run 8/20] 04acbaef-382d-4286-9cc8-ffa1e05482e7: ok=True cer_norm=0.9514148424986653
[run 9/20] 062a64cb-6c04-45f0-bce9-7b4558198792: ok=True cer_norm=0.99179580674567
[run 10/20] 066215aa-688f-4aab-a39f-37825ac460b9: ok=True cer_norm=0.8956924172303311
[run 11/20] 06d2ddda-9abd-4ace-a00f-33a9138ff593: ok=True cer_norm=0.99

## 12. Result Validation

In [18]:
import json

raw_files = sorted((run_dir / "raw_outputs").glob("*.json"))
metrics = json.loads((run_dir / "metrics.json").read_text())

for artifact in ("raw_outputs", "metrics.json", "config.yaml", "run_log.md"):
    assert (run_dir / artifact).exists(), f"missing artifact: {artifact}"
assert len(raw_files) == LIMIT, f"expected {LIMIT} raw outputs, got {len(raw_files)}"
assert metrics["sample_count"] == LIMIT
assert metrics["successful_samples"] + metrics["failed_samples"] == LIMIT
if metrics.get("accelerator_device") != "cuda":
    raise SystemExit("GPU-FIRST VIOLATION: accelerator_device != cuda")
assert metrics["micro"]["cer_normalized"] >= 0
assert metrics["dataset"] == "misraj"
assert metrics["engine"] == "docling"

report["Sample count"] = metrics["sample_count"]
report["Successful samples"] = metrics["successful_samples"]
report["Failed samples"] = metrics["failed_samples"]
report["Accelerator"] = metrics["accelerator_device"].upper()
report["Result validation"] = "PASS"
print("validation summary:")
print(f"  run dir          : {run_dir.name}")
print(f"  raw outputs      : {len(raw_files)}")
print(f"  ok / failed      : {metrics['successful_samples']} / {metrics['failed_samples']}")
print(f"  micro CER (norm) : {metrics['micro']['cer_normalized']:.4f}")
print(f"  accelerator      : {metrics['accelerator_device']}")

validation summary:
  run dir          : 20260826-051654
  raw outputs      : 20
  ok / failed      : 20 / 0
  micro CER (norm) : 0.9666
  accelerator      : cuda


## 13. Result Archival

In [19]:
artifact = DIST / f"external_result_{run_dir.name}.tar.gz"
assert not artifact.exists(), f"refusing to overwrite existing artifact: {artifact}"
sh(f"tar -czf {artifact} -C {RESULTS} {run_dir.name}")
assert artifact.is_file(), "artifact missing after archiving"
artifact_sha = sh(f"sha256sum {artifact}").split()[0]
print("artifact sha256:", artifact_sha)
report["Drive artifact"] = "PASS"
print("ARTIFACT WRITTEN TO:")
print(artifact)

artifact sha256: 037d39bfbd7d5acbb483dc18d72ceec78da6957ea2faca1428042b0bdcc1fe16
ARTIFACT WRITTEN TO:
/content/drive/MyDrive/ocrbench/dist/external_result_20260826-051654.tar.gz


## 14. Final Summary

In [23]:
line = "=" * 42

required_report_keys = [
    "Repository commit",
    "Python 3.12 environment",
    "Torch CUDA",
    "gpu",
    "Docling",
    "Dataset archive hash",
    "Dataset manifest integrity",
    "Benchmark execution",
    "Sample count",
    "Successful samples",
    "Failed samples",
    "Accelerator",
    "Result validation",
    "Drive artifact",
]

missing = [key for key in required_report_keys if key not in report]
assert not missing, f"Final summary cannot run; missing report keys: {missing}"

def get(key):
    value = report[key]
    return value if isinstance(value, str) else str(value)

print(line)
print("EXTERNAL GPU SMOKE VALIDATION")
print(line)

print("Repository commit:", get("Repository commit"), f"({PINNED_COMMIT[:12]})")
print("Python 3.12 environment:", get("Python 3.12 environment"))
print("Torch CUDA:", get("Torch CUDA"))
print("GPU:", get("gpu"))
print("Docling:", get("Docling"))
print("Dataset archive hash:", get("Dataset archive hash"))
print("Dataset manifest integrity:", get("Dataset manifest integrity"))
print("Benchmark execution:", get("Benchmark execution"))
print("Sample count:", get("Sample count"))
print("Successful samples:", get("Successful samples"))
print("Failed samples:", get("Failed samples"))
print("Accelerator:", get("Accelerator"))
print("Result validation:", get("Result validation"))
print("Drive artifact:", get("Drive artifact"))

print(line)

EXTERNAL GPU SMOKE VALIDATION
Repository commit: PASS (2c6975c3d5b2)
Python 3.12 environment: Python 3.12.14
Torch CUDA: PASS
GPU: Tesla T4
Docling: PASS
Dataset archive hash: PASS
Dataset manifest integrity: PASS (2 files)
Benchmark execution: PASS
Sample count: 20
Successful samples: 20
Failed samples: 0
Accelerator: CUDA
Result validation: PASS
Drive artifact: PASS
